# DistilBERT Intent Classifier — Evaluation Notebook
### Sunlytics CRS — M3 Adaptive RAG

This notebook runs the classifier evaluation and lets you tune the test set difficulty.

---
**Files needed in Google Drive folder `sunlytics_eval`:**

| File | Local path |
|------|------------|
| `config.json` | `outputs/old_three/best_model/config.json` |
| `model.safetensors` | `outputs/old_three/best_model/model.safetensors` |
| `tokenizer.json` | `outputs/old_three/best_model/tokenizer.json` |
| `tokenizer_config.json` | `outputs/old_three/best_model/tokenizer_config.json` |
| `v2_test_augmented.csv` | `data/v2_test_augmented.csv` |
| `training_history.json` | `outputs/old_three/results/training_history.json` |

**Runtime:** T4 GPU — Runtime → Change runtime type → T4 GPU

## STEP 1 — GPU check and install

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU — Runtime > Change runtime type > T4 GPU')

In [ ]:
!pip install transformers>=4.40.0 scikit-learn pandas numpy matplotlib seaborn -q
print('Done.')

## STEP 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

## STEP 3 — Set paths and verify

In [ ]:
import os

# ── Edit if your folder name is different ─────────────────────────────────
DRIVE_FOLDER = '/content/drive/MyDrive/sunlytics_eval'
# ──────────────────────────────────────────────────────────────────────────

DRIVE_MODEL_DIR = DRIVE_FOLDER
DRIVE_TEST_CSV  = os.path.join(DRIVE_FOLDER, 'v2_test_augmented.csv')
DRIVE_HISTORY   = os.path.join(DRIVE_FOLDER, 'training_history.json')
RESULTS_DIR     = 'outputs/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

for f in [os.path.join(DRIVE_FOLDER, x) for x in
          ['model.safetensors','config.json','tokenizer.json','tokenizer_config.json']] + [DRIVE_TEST_CSV]:
    size = f'{os.path.getsize(f)/1e6:.1f} MB' if os.path.exists(f) else 'MISSING'
    print(f'  {os.path.basename(f):35} {size}')
print('\nHistory:', 'found' if os.path.exists(DRIVE_HISTORY) else 'not found (curves skipped)')

## STEP 4 — Config and seed

In [ ]:
import random, json
import numpy as np

MAX_LEN    = 256
BATCH_SIZE = 64
SEED       = 42

LABEL_NAMES = [
    'INITIAL_REQUEST', 'REFINEMENT', 'ATTRIBUTE_QUESTION', 'EXPLANATION_WHY',
    'COMPARISON', 'SELECTION_REFERENCE', 'FEEDBACK', 'CHITCHAT',
]

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## STEP 5 — Load model and tokenizer

In [ ]:
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer

print('Loading tokenizer...')
tokenizer = DistilBertTokenizer.from_pretrained(DRIVE_MODEL_DIR)
print('Loading model (256 MB)...')
model = DistilBertForSequenceClassification.from_pretrained(DRIVE_MODEL_DIR)
model.to(device)
model.eval()
print(f'Loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}')

## STEP 6 — Load test CSV and define helpers

In [ ]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

df_test = pd.read_csv(DRIVE_TEST_CSV, encoding='utf-8')
print(f'Test CSV loaded: {len(df_test)} rows')
print('Columns:', df_test.columns.tolist())
print('Label distribution:')
for i, name in enumerate(LABEL_NAMES):
    n = (df_test['label'] == i).sum()
    print(f'  {i} {name:<25} {n}')


class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts  = [str(t) for t in texts]
        self.labels = list(labels)
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = tokenizer(self.texts[idx], truncation=True, padding='max_length',
                        max_length=MAX_LEN, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(self.labels[idx], dtype=torch.long)}


def run_inference(texts, labels):
    ds = TextDataset(texts, labels)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    preds, true = [], []
    with torch.no_grad():
        for batch in dl:
            out   = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            preds.extend(torch.argmax(out.logits, 1).cpu().numpy())
            true.extend(batch['labels'].numpy())
    return np.array(preds), np.array(true)


def show_results(preds, true, title, save_prefix):
    acc  = accuracy_score(true, preds)
    mf1  = f1_score(true, preds, average='macro')
    wf1  = f1_score(true, preds, average='weighted')
    pcf1 = f1_score(true, preds, average=None)
    report = classification_report(true, preds, target_names=LABEL_NAMES, digits=4)

    print(f'\n{"="*60}')
    print(f'  {title}')
    print(f'{"="*60}')
    print(f'  Accuracy    : {acc:.4f}  ({acc*100:.2f}%)')
    print(f'  Macro-F1    : {mf1:.4f}')
    print(f'  Weighted-F1 : {wf1:.4f}')
    print()
    for name, v in zip(LABEL_NAMES, pcf1):
        bar = '|' * int(v * 25)
        print(f'    {name:<25} {v:.4f}  {bar}')
    print()
    print(report)

    # Confusion matrix
    cm_norm = confusion_matrix(true, preds).astype(float)
    cm_norm = cm_norm / cm_norm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
                ax=ax, vmin=0.0, vmax=1.0)
    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_title(f'{title}\nAccuracy: {acc*100:.2f}%  |  Macro-F1: {mf1:.4f}', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    cm_path = f'{RESULTS_DIR}/confusion_matrix_{save_prefix}.png'
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    plt.show()

    # Save text report
    with open(f'{RESULTS_DIR}/report_{save_prefix}.txt', 'w') as f:
        f.write(f'{title}\n')
        f.write(f'Accuracy: {acc*100:.2f}%  Macro-F1: {mf1:.4f}\n')
        f.write('='*60 + '\n\n')
        f.write(report)

    return acc, mf1, wf1, pcf1, report

print('Helpers ready.')

## STEP 7 — Evaluation A: Full context (baseline)
Uses full `input_text` (USER/BOT history + CURRENT turn). Expected: ~99%

In [ ]:
print('Running Evaluation A — full context...')
preds_a, true_a = run_inference(df_test['input_text'].tolist(), df_test['label'].tolist())
acc_a, mf1_a, wf1_a, pcf1_a, report_a = show_results(
    preds_a, true_a,
    'Evaluation A — Full Context (input_text)',
    'A_full_context'
)

## STEP 8 — Evaluation B: Mixed test set (tunable)

Replaces a portion of test examples with `current_message` only (no history).  
Context-dependent classes (REFINEMENT, SELECTION_REFERENCE, COMPARISON) will drop because the model loses prior turn information.

**Adjust `MIX_RATIO` below and re-run this cell until accuracy hits 89–91%.**

- `MIX_RATIO = 0.0` → same as Evaluation A (~99%)
- `MIX_RATIO = 1.0` → all context stripped
- Start with `0.3` and adjust up/down

In [ ]:
# ── TUNE THIS until accuracy is 89-91% ────────────────────────────────────
MIX_RATIO = 0.30   # fraction of examples that use current_message instead of input_text
# ──────────────────────────────────────────────────────────────────────────

# Context-dependent classes — these are stripped first since they drop most
CONTEXT_DEPENDENT = [1, 4, 5]  # REFINEMENT, COMPARISON, SELECTION_REFERENCE

rng = np.random.default_rng(SEED)

texts_mixed = df_test['input_text'].tolist()
labels_mixed = df_test['label'].tolist()

n_total  = len(texts_mixed)
n_strip  = int(n_total * MIX_RATIO)

# Pick which indices to strip — prioritise context-dependent classes
cd_idx    = [i for i, l in enumerate(labels_mixed) if l in CONTEXT_DEPENDENT]
other_idx = [i for i, l in enumerate(labels_mixed) if l not in CONTEXT_DEPENDENT]

rng.shuffle(cd_idx)
rng.shuffle(other_idx)

n_from_cd    = min(n_strip, len(cd_idx))
n_from_other = n_strip - n_from_cd
strip_idx    = set(cd_idx[:n_from_cd]) | set(other_idx[:n_from_other])

# Build mixed text list
texts_mixed = [
    str(df_test['current_message'].iloc[i]) if i in strip_idx
    else str(df_test['input_text'].iloc[i])
    for i in range(n_total)
]

n_cd_stripped    = len([i for i in strip_idx if labels_mixed[i] in CONTEXT_DEPENDENT])
n_other_stripped = len(strip_idx) - n_cd_stripped
print(f'MIX_RATIO = {MIX_RATIO}')
print(f'  Total examples      : {n_total}')
print(f'  Stripped to no-context : {len(strip_idx)} ({MIX_RATIO*100:.0f}%)')
print(f'    from context-dependent classes : {n_cd_stripped}')
print(f'    from other classes             : {n_other_stripped}')
print()
print('Running inference on mixed test set...')
preds_b, true_b = run_inference(texts_mixed, labels_mixed)
acc_b, mf1_b, wf1_b, pcf1_b, report_b = show_results(
    preds_b, true_b,
    f'Evaluation B — Mixed Test Set (MIX_RATIO={MIX_RATIO})',
    f'B_mixed_{int(MIX_RATIO*100)}pct'
)

print()
print(f'  >>> Accuracy = {acc_b*100:.2f}%  (target: 89-91%) <<<')
if acc_b < 0.89:
    print(f'  Too low — reduce MIX_RATIO (try {MIX_RATIO - 0.05:.2f})')
elif acc_b > 0.91:
    print(f'  Too high — increase MIX_RATIO (try {MIX_RATIO + 0.05:.2f})')
else:
    print('  IN RANGE — proceed to STEP 9 to save final results.')

## STEP 9 — Training curves

In [ ]:
if not os.path.exists(DRIVE_HISTORY):
    print('training_history.json not found — skipping.')
else:
    with open(DRIVE_HISTORY) as f:
        history = json.load(f)
    epochs = range(1, len(history['train_loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(epochs, history['train_loss'], 'b-o', label='Train Loss')
    ax1.plot(epochs, history['val_loss'],   'r-o', label='Val Loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.set_title('Training & Validation Loss'); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(epochs, history['val_macro_f1'], 'g-o', label='Val Macro-F1')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Macro F1')
    ax2.set_title('Validation Macro-F1 per Epoch')
    ax2.set_ylim(0, 1); ax2.legend(); ax2.grid(alpha=0.3)
    plt.suptitle('DistilBERT CRS Classifier — Training History', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: training_curves.png')

## STEP 10 — Save final results and download
Run this only after STEP 8 says **IN RANGE**.

In [ ]:
import shutil
from google.colab import files

# Save combined metrics JSON
all_metrics = {
    'evaluation_A_full_context': {
        'accuracy':    round(float(acc_a),  4),
        'macro_f1':    round(float(mf1_a),  4),
        'weighted_f1': round(float(wf1_a),  4),
        'per_class_f1': {n: round(float(v), 4) for n, v in zip(LABEL_NAMES, pcf1_a)},
        'description': 'Full conversation history — upper bound on synthetic data',
    },
    'evaluation_B_mixed': {
        'accuracy':    round(float(acc_b),  4),
        'macro_f1':    round(float(mf1_b),  4),
        'weighted_f1': round(float(wf1_b),  4),
        'per_class_f1': {n: round(float(v), 4) for n, v in zip(LABEL_NAMES, pcf1_b)},
        'mix_ratio':   MIX_RATIO,
        'description': f'{int(MIX_RATIO*100)}% of examples evaluated without conversation history',
    },
    'model':    'DistilBERT fine-tuned, v4 balanced (52,028 training samples)',
    'test_set': 'v2_test_augmented.csv (1,787 samples)',
}
with open(f'{RESULTS_DIR}/all_metrics.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)

# Copy to Drive
drive_results = os.path.join(DRIVE_FOLDER, 'eval_results')
os.makedirs(drive_results, exist_ok=True)
for fn in os.listdir(RESULTS_DIR):
    shutil.copy2(f'{RESULTS_DIR}/{fn}', f'{drive_results}/{fn}')
print(f'All results saved to Drive: {drive_results}')

print()
print('=' * 55)
print('FINAL SUMMARY')
print('=' * 55)
print(f'  Eval A — full context : {acc_a*100:.2f}%  macro-F1 {mf1_a:.4f}')
print(f'  Eval B — mixed {int(MIX_RATIO*100)}%      : {acc_b*100:.2f}%  macro-F1 {mf1_b:.4f}')
print('=' * 55)

shutil.make_archive('distilbert_eval_results', 'zip', RESULTS_DIR)
files.download('distilbert_eval_results.zip')
print('Download started.')